# Chapter 2: Turn domain work into executable contracts

Companion notebook to *Agent Harness Engineering* by Praneeth Paikray.

Run every cell in order from a fresh kernel. Sections 1 to 9 use the Python standard library only. Section 10 is optional and points the note reader at a real hosted model; it skips itself cleanly when no model is configured (set-up is in the Chapter 1 notebook, Section 8).

You will load the domain manifest, the numeric evaluation rule, the stand-in note reader and the grounding contract around it, the records and twelve fixtures behind Table 2.3, and the evaluator. Then the three exercises, with solutions. Then the same three clinical notes through a live model.

Every patient, protocol, note, marker, threshold, and date here is fictional. They show how software behaves and mean nothing clinically.

## 1. Check the environment

In [ ]:
import sys

assert sys.version_info >= (3, 10), "Use Python 3.10 or newer."
print("Python", ".".join(map(str, sys.version_info[:3])))

## 2. Listing 2.1: types

Two kinds of rule share a protocol version's `criteria` tuple: `Criterion` compares a number against a range; `Exclusion` is stated in prose and must be checked against clinical notes. `Evidence` and `Note` are the two things a patient record can hold.

In [ ]:
from dataclasses import dataclass
from datetime import date

@dataclass(frozen=True)
class Criterion:             # a numeric rule
    id: str
    field: str
    lower: float
    upper: float
    unit: str
    max_age_days: int | None = None   # None: any age
@dataclass(frozen=True)
class Exclusion:             # a rule stated in prose
    id: str
    field: str               # what the model must look for
    window_days: int
@dataclass(frozen=True)
class ProtocolVersion:
    study: str
    version: int
    effective_from: date
    criteria: tuple          # Criterion or Exclusion
@dataclass(frozen=True)
class Evidence:
    id: str
    field: str
    value: float             # 1.0 present, 0.0 absent
    unit: str
    observed: date
@dataclass(frozen=True)
class Note:
    id: str
    written: date
    text: str

## 3. Listing 2.2: the manifest and version selection

The manifest is data. The book prints it as Python so you can read it; the cell after this one writes it to a JSON file and reads it back into the same dataclasses, which is how the repository stores it from Chapter 3 on.

In [ ]:
AGE = Criterion("age", "age", 18, 65, "years")
PROTOCOLS = [
    ProtocolVersion("T004", 2, date(2026, 1, 15), (AGE,
        Criterion("marker", "marker", 3, 7, "ng/mL"))),
    ProtocolVersion("T004", 3, date(2026, 8, 1), (AGE,
        Criterion("marker", "marker", 4, 8, "ng/mL", 90),
        Exclusion("anticoag", "anticoagulant", 180))),
]
ASSIGNED = {"reviewer-a": {"T004"}, "reviewer-b": {"T009"}}
VOCABULARY = {"age", "marker", "anticoagulant"}

class ContractError(Exception):
    pass

def select_protocol(study, version, on):
    live = [p for p in PROTOCOLS
            if p.study == study and p.effective_from <= on]
    if version == "current" and live:
        return max(live, key=lambda p: p.version)
    for p in live:
        if p.version == version:
            return p
    raise ContractError(f"{study} v{version} not in effect "
                        f"on {on}")

### The manifest as a file

Nothing in the harness depends on the manifest being Python. Here it goes out as JSON and comes back as the same frozen objects.

In [ ]:
import json, tempfile, pathlib

def to_json(protocols):
    out = []
    for p in protocols:
        rules = []
        for r in p.criteria:
            d = {"kind": type(r).__name__, **r.__dict__}
            rules.append(d)
        out.append({"study": p.study, "version": p.version,
                    "effective_from": p.effective_from.isoformat(), "criteria": rules})
    return json.dumps(out, indent=1)

def from_json(text):
    kinds = {"Criterion": Criterion, "Exclusion": Exclusion}
    result = []
    for p in json.loads(text):
        rules = tuple(kinds[r.pop("kind")](**r) for r in p["criteria"])
        result.append(ProtocolVersion(p["study"], p["version"],
                                      date.fromisoformat(p["effective_from"]), rules))
    return result

path = pathlib.Path(tempfile.gettempdir()) / "t004_manifest.json"
path.write_text(to_json(PROTOCOLS))
reloaded = from_json(path.read_text())
assert reloaded == PROTOCOLS
print(f"Manifest round-tripped through {path.name}: {len(reloaded)} versions, identical.")
print(path.read_text()[:400], "...")

## 4. Records, notes, and fixtures (Table 2.3)

Eight synthetic patients, three clinical notes, one dishonest model (`liar`), and twelve fixtures. Each fixture is a frozen request, the expected packet fields, the expected status of every rule, and, where a note is involved, the model to use. All of it was written before the code ran.

In [ ]:
D = date  # shorthand for the fixture tables
RECORDS = {
    "P017": [Evidence("e01", "age", 47, "years", D(2026, 9, 1))],
    "P023": [Evidence("e02", "age", 52, "years", D(2026, 9, 1)),
             Evidence("e03", "marker", 5, "ng/mL", D(2025, 6, 3))],
    "P031": [Evidence("e04", "age", 39, "years", D(2026, 9, 1)),
             Evidence("e05", "marker", 5, "ng/mL", D(2026, 8, 20))],
    "P032": [Evidence("e06", "age", 44, "years", D(2026, 9, 1)),
             Evidence("e07", "marker", 3.5, "ng/mL", D(2026, 8, 22))],
    "P034": [Evidence("e10", "age", 29, "years", D(2026, 9, 1)),
             Evidence("e11", "marker", 5, "nmol/L", D(2026, 8, 25))],
    "P036": [Evidence("e15", "age", 58, "years", D(2026, 9, 1)),
             Evidence("e16", "marker", 6, "ng/mL", D(2026, 8, 28))],
    "P037": [Evidence("e17", "age", 41, "years", D(2026, 9, 1)),
             Evidence("e18", "marker", 5.5, "ng/mL", D(2026, 8, 27))],
    "P038": [Evidence("e19", "age", 36, "years", D(2026, 9, 1)),
             Evidence("e20", "marker", 6.2, "ng/mL", D(2026, 8, 29))],
}
NOTES = {
    "P036": [Note("n01", D(2026, 7, 2),
                  "Atrial fibrillation. Started warfarin 5 mg daily on "
                  "2026-07-02. Follow-up in six weeks.")],
    "P037": [Note("n02", D(2026, 8, 30),
                  "No anticoagulant therapy in the past year, reviewed "
                  "2026-08-30. Continues metformin.")],
    "P038": [Note("n03", D(2026, 8, 29),
                  "Palpitations reported. Discussed starting apixaban at "
                  "the next visit pending cardiology review.")],
}

def liar(context):
    # A model that invents a quotation. Grounding must catch it.
    return {"field": context["field"], "status": "present",
            "observed": "2026-08-01", "quote": "Currently on warfarin"}

def request(patient, version, on=D(2026, 9, 1), by="reviewer-a"):
    return {"patient": patient, "study": "T004", "version": version,
            "on": on, "by": by}

FIXTURES = [
    {"id": "F01", "shape": "all criteria met (v2)",
     "request": request("P031", 2), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met"}},
    {"id": "F02", "shape": "marker missing",
     "request": request("P017", 2), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "unknown"}},
    {"id": "F03", "shape": "marker stale under v3",
     "request": request("P023", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "unknown", "anticoag": "unknown"}},
    {"id": "F04", "shape": "marker in the wrong unit",
     "request": request("P034", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "unknown", "anticoag": "unknown"}},
    {"id": "F05", "shape": "version not yet in effect",
     "request": request("P031", 3, on=D(2026, 7, 15)),
     "expect": {"status": "refused",
                "reason": "T004 v3 not in effect on 2026-07-15"}},
    {"id": "F06", "shape": "'current' resolves to v3",
     "request": request("P032", "current"),
     "expect": {"status": "prepared", "protocol": "T004 v3"},
     "criteria": {"age": "met", "marker": "not met", "anticoag": "unknown"}},
    {"id": "F07", "shape": "reviewer not assigned to study",
     "request": request("P031", 2, by="reviewer-b"),
     "expect": {"status": "refused",
                "reason": "reviewer-b is not assigned to T004"}},
    {"id": "F08", "shape": "no such patient record",
     "request": request("P999", 2),
     "expect": {"status": "refused", "reason": "no record: P999"}},
    {"id": "F09", "shape": "note states anticoagulant started in window",
     "request": request("P036", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met", "anticoag": "not met"}},
    {"id": "F10", "shape": "note states absence, with a date",
     "request": request("P037", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met", "anticoag": "met"}},
    {"id": "F11", "shape": "note mentions the drug but no date",
     "request": request("P038", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met", "anticoag": "unknown"}},
    {"id": "F12", "shape": "model invents a quotation",
     "request": request("P036", 3), "model": liar,
     "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met", "anticoag": "unknown"}},
]

## 5. Listing 2.3: evaluate the numbers

Four exits, each with its reason and the evidence item it looked at. Two fresh readings that disagree are not handled; the current code takes the more recent. That is Exercise 2.2.

In [ ]:
def evaluate(criterion, evidence, on):
    items = [e for e in evidence
             if e.field == criterion.field]
    if not items:
        return "unknown", "no value recorded", None
    odd = [e for e in items if e.unit != criterion.unit]
    if odd:
        e = odd[0]
        return "unknown", f"unit is {e.unit}", e.id
    limit = criterion.max_age_days
    if limit is not None:
        fresh = [e for e in items
                 if (on - e.observed).days <= limit]
        if not fresh:
            e = max(items, key=lambda e: e.observed)
            return "unknown", f"stale: {e.observed}", e.id
        items = fresh
    e = max(items, key=lambda e: e.observed)
    inside = criterion.lower <= e.value <= criterion.upper
    return ("met" if inside else "not met",
            f"{e.value} {e.unit} on {e.observed}", e.id)

## 6. Listing 2.4: a stand-in reader

The model's first real job in the book is to read a note and return a statement: `present`, `absent`, `unclear`, or `none`; the sentence it rests on, quoted; and the date it applies to. This stand-in finds a cue word and returns that sentence with any ISO date in it. It is deliberately crude and deliberately deterministic. Section 10 replaces it with a real model without touching anything else.

In [ ]:
import re
CUES = ("warfarin", "apixaban", "anticoagulant")

def fake_reader(context):
    # Stand-in for a model reading a note: finds a cue word
    # and returns that sentence with any ISO date in it.
    # A real model reads better; the contract is the same.
    for sentence in context["note"].split(". "):
        low = sentence.lower()
        if any(cue in low for cue in CUES):
            when = re.search(r"\d{4}-\d{2}-\d{2}", sentence)
            absent = low.startswith("no ")
            status = "absent" if absent else "present"
            return {"field": context["field"],
                    "status": status,
                    "observed": when and when.group(),
                    "quote": sentence.strip().rstrip(".")}
    return {"field": context["field"], "status": "none",
            "observed": None, "quote": ""}

## 7. Listing 2.5: the contract around the reader

`ground` refuses a proposal whose field is outside the vocabulary, whose quotation is not in the note, that is unclear, or that has no date. Only a grounded proposal becomes an `Evidence` item, and then `evaluate_exclusion` applies the window exactly as `evaluate` applies freshness.

In [ ]:
def ground(proposal, note):
    # The model proposes. The note and the manifest decide.
    if proposal["field"] not in VOCABULARY:
        return "field not in vocabulary"
    if proposal["status"] == "none":
        return "no statement in note"
    if proposal["quote"] not in note.text:
        return "quote not found in note"
    if proposal["status"] == "unclear":
        return "statement unclear; needs review"
    if not proposal.get("observed"):
        return "statement has no date"
    return None

def read_note(model, note, field):
    proposal = model({"note": note.text, "field": field})
    problem = ground(proposal, note)
    if problem:
        return None, problem
    present = proposal["status"] == "present"
    when = date.fromisoformat(proposal["observed"])
    return Evidence(note.id, field, 1.0 if present else 0.0,
                    "statement", when), None

def evaluate_exclusion(rule, notes, on, model):
    found, why = [], "no statement in notes"
    for note in notes:
        item, problem = read_note(model, note, rule.field)
        if item is None:
            why = f"{note.id}: {problem}"
        else:
            found.append(item)
    recent = [f for f in found
              if (on - f.observed).days <= rule.window_days]
    if any(f.value for f in recent):
        f = next(f for f in recent if f.value)
        return "not met", f"present {f.observed}", f.id
    if recent:
        f = recent[-1]
        return "met", f"absence stated {f.observed}", f.id
    return "unknown", why, None

## 8. Listing 2.6: validate, prepare, and run the evaluator

Expected output: twelve `pass` lines. Model calls are 0 for every refused request and every version-2 request (version 2 has no text rule), and 1 for each version-3 request with a note to read.

In [ ]:
def validate(req):
    if req["patient"] not in RECORDS:
        raise ContractError(f"no record: {req['patient']}")
    if req["study"] not in ASSIGNED.get(req["by"], set()):
        raise ContractError(f"{req['by']} is not assigned "
                            f"to {req['study']}")
    return select_protocol(req["study"], req["version"],
                           req["on"])

def prepare(req, model=fake_reader):
    try:
        protocol = validate(req)
    except ContractError as why:
        return {"status": "refused", "reason": str(why)}
    evidence = RECORDS[req["patient"]]
    notes = NOTES.get(req["patient"], [])
    results = {}
    for rule in protocol.criteria:
        if isinstance(rule, Exclusion):
            results[rule.id] = evaluate_exclusion(
                rule, notes, req["on"], model)
        else:
            results[rule.id] = evaluate(rule, evidence,
                                        req["on"])
    name = f"{protocol.study} v{protocol.version}"
    return {"status": "prepared", "protocol": name,
            "criteria": results, "eligible": None,
            "review": "pending"}

for fx in FIXTURES:
    calls = []
    def counted(context, m=fx.get("model", fake_reader)):
        calls.append(context)
        return m(context)
    got = prepare(fx["request"], model=counted)
    found = got.get("criteria", {}).items()
    seen = {k: v[0] for k, v in found}
    want = fx["expect"].items()
    ok = (all(got.get(k) == v for k, v in want)
          and seen == fx.get("criteria", {}))
    print(fx["id"], "pass" if ok else "FAIL", got["status"],
          "model calls:", len(calls))

The checks below are what the chapter asserts about this run.

In [ ]:
outcomes = {fx["id"]: prepare(fx["request"], model=fx.get("model", fake_reader)) for fx in FIXTURES}
assert sum(o["status"] == "refused" for o in outcomes.values()) == 3
assert outcomes["F09"]["criteria"]["anticoag"][:2] == ("not met", "present 2026-07-02")
assert outcomes["F10"]["criteria"]["anticoag"][0] == "met"
assert outcomes["F11"]["criteria"]["anticoag"][1].endswith("statement has no date")
assert outcomes["F12"]["criteria"]["anticoag"][1].endswith("quote not found in note")
print("Chapter 2 behaves as described.")
for fid, o in outcomes.items():
    if o["status"] == "prepared":
        print(fid, o["protocol"], {k: v[:2] for k, v in o["criteria"].items()})
    else:
        print(fid, o)

## 9. Exercises

### Exercise 2.1: same evidence, two versions

Prepare P023 under version 2 and under version 3 on the same date and compare the marker result.

In [ ]:
for version in (2, 3):
    p = prepare(request("P023", version))
    print(p["protocol"], p["criteria"]["marker"])

**Solution 2.1.** Under version 2 the marker is `met`, citing `e03` (5 ng/mL, June 2025), because version 2 has no freshness limit. Under version 3 the same item is `unknown`, `stale: 2025-06-03`. The patient did not change; the evidence contract did. That is why every packet names its protocol version and why every status carries the evidence id: a packet that says only "marker: met" is not checkable a month later.

### Exercise 2.2: two fresh readings that disagree

Give one patient two fresh marker readings that disagree. Decide what the coordinator would want, write the fixture first, then change `evaluate`.

In [ ]:
RECORDS["P035"] = [Evidence("e12", "age", 51, "years", D(2026, 9, 1)),
                   Evidence("e13", "marker", 5, "ng/mL", D(2026, 8, 10)),
                   Evidence("e14", "marker", 6.5, "ng/mL", D(2026, 8, 24))]
FIXTURES.append({"id": "F13", "shape": "two fresh readings disagree",
                 "request": request("P035", 3), "expect": {"status": "prepared"},
                 "criteria": {"age": "met", "marker": "unknown", "anticoag": "unknown"}})

print("before:", prepare(request("P035", 3))["criteria"]["marker"])

def evaluate(criterion, evidence, on):
    items = [e for e in evidence if e.field == criterion.field]
    if not items:
        return "unknown", "no value recorded", None
    odd = [e for e in items if e.unit != criterion.unit]
    if odd:
        return "unknown", f"unit is {odd[0].unit}", odd[0].id
    limit = criterion.max_age_days
    if limit is not None:
        fresh = [e for e in items if (on - e.observed).days <= limit]
        if not fresh:
            e = max(items, key=lambda e: e.observed)
            return "unknown", f"stale: {e.observed}", e.id
        items = fresh
    if len({e.value for e in items}) > 1:
        return "unknown", "conflicting values", None
    e = max(items, key=lambda e: e.observed)
    inside = criterion.lower <= e.value <= criterion.upper
    return ("met" if inside else "not met", f"{e.value} {e.unit} on {e.observed}", e.id)

print("after: ", prepare(request("P035", 3))["criteria"]["marker"])

**Solution 2.2.** The coordinator said "I flag it and ask the site." Picking the more recent reading silently is a decision the harness is not entitled to make, so the fixture expects `unknown`, and the new exit returns `conflicting values`. Note the order: the conflict check runs after the freshness filter, so a stale disagreeing reading does not block a single fresh one.

### Exercise 2.3: a misreading the contract cannot catch

Write a note the stand-in reader gets wrong but grounding cannot catch.

In [ ]:
NOTES["P031"] = [Note("n04", D(2026, 7, 10),
                      "Warfarin was considered on 2026-07-10 and rejected in favour "
                      "of compression therapy. No anticoagulant started.")]
p = prepare(request("P031", 3))
print(p["criteria"]["anticoag"])
del NOTES["P031"]

**Solution 2.3.** The stand-in returns the first cue sentence, "Warfarin was considered on 2026-07-10 and rejected...", as `present` with a valid date. The quotation is verbatim, the date parses, the field is in the vocabulary: `ground` has no grounds to refuse, and the exclusion comes out `not met`. Wrong.

The packet field that saves the reviewer is the evidence id `n04` together with the reason string: the reviewer follows it to the note and reads the sentence the model read. That is the escalation path, and it is why review stays pending even on a confident `not met`. To *measure* how often this happens you need labelled notes (a human-decided status per note), a set of them held out from whatever you used to tune the prompt, and repeated trials, which is Chapter 9. A real model does far better than the stand-in on this sentence, as Section 10 shows, but "far better" is a number to be measured, not assumed.

## 10. Optional: the same contract with a real reader

Configure a model as in the Chapter 1 notebook (Section 8): inside a Databricks workspace credentials are automatic; locally, set `DATABRICKS_HOST`, `DATABRICKS_TOKEN`, and optionally `MODEL_ENDPOINT`. Then `live_reader` below is Listing 1.3's adapter with a different question. The harness (`ground`, `read_note`, `evaluate_exclusion`, `prepare`) does not change.

In [ ]:
import os, urllib.request, urllib.error, json

try:
    from databricks.sdk import WorkspaceClient
    _w = WorkspaceClient()
    os.environ.setdefault("DATABRICKS_HOST", _w.config.host)
    _auth = _w.config.authenticate().get("Authorization", "")
    if _auth.startswith("Bearer "):
        os.environ.setdefault("DATABRICKS_TOKEN", _auth.split(" ", 1)[1])
except Exception:
    pass
LIVE = bool(os.environ.get("DATABRICKS_HOST") and os.environ.get("DATABRICKS_TOKEN"))
ENDPOINT = os.environ.get("MODEL_ENDPOINT", "databricks-gpt-5-6-luna")

def live_reader(context):
    host = os.environ["DATABRICKS_HOST"].rstrip("/")
    ask = ("You are extracting one fact from a clinical note for a screening "
           f"harness. Field: {context['field']}. Reply with JSON only: "
           '{"field": "<the field>", "status": "present"|"absent"|"unclear"|"none", '
           '"observed": "YYYY-MM-DD" or null, "quote": "<a verbatim sentence from the note>"}. '
           "Use present if the note states the therapy is or was being taken, absent if it "
           "states it is not, unclear if the note discusses it without saying, none if the "
           "note does not mention it. The quote must be copied exactly from the note.\n\nNOTE:\n"
           + context["note"])
    body = json.dumps({"model": ENDPOINT, "messages": [{"role": "user", "content": ask}]}).encode()
    req = urllib.request.Request(host + "/serving-endpoints/chat/completions", body,
                                 {"Content-Type": "application/json",
                                  "Authorization": "Bearer " + os.environ["DATABRICKS_TOKEN"]})
    with urllib.request.urlopen(req, timeout=60) as reply:
        text = json.load(reply)["choices"][0]["message"]["content"]
    try:
        return json.loads(text.strip().strip("`").removeprefix("json").strip())
    except ValueError:
        return {"field": context["field"], "status": "unclear", "observed": None, "quote": text[:80]}

print("Live model available:", LIVE, "| endpoint:", ENDPOINT if LIVE else "(none)")

### Three notes, two readers

For each note: what the stand-in proposed, what the live model proposed, and what `ground` said about each. The exclusion status for the patient follows. The Exercise 2.3 note is included because it is the one the stand-in gets wrong.

In [ ]:
import datetime
probes = [("n01", NOTES["P036"][0]), ("n02", NOTES["P037"][0]), ("n03", NOTES["P038"][0]),
          ("n04", Note("n04", D(2026, 7, 10), "Warfarin was considered on 2026-07-10 and rejected "
                       "in favour of compression therapy. No anticoagulant started."))]
if LIVE:
    print(f"{datetime.date.today()}  endpoint={ENDPOINT}")
    for nid, note in probes:
        for name, reader in (("stand-in", fake_reader), ("live", live_reader)):
            try:
                prop = reader({"note": note.text, "field": "anticoagulant"})
                verdict = ground(prop, note) or "grounded"
                print(f"{nid} {name:8s} status={prop.get('status')!s:8s} observed={prop.get('observed')!s:11s} "
                      f"-> {verdict}  quote={prop.get('quote')!r}")
            except urllib.error.HTTPError as error:
                print(nid, name, "HTTP", error.code, error.read().decode()[:200])
else:
    print("Skipped: no live model configured.")

### What to look for

The live model will usually mark n04 `absent` or `unclear` where the stand-in said `present`, and it may quote a different sentence. Whatever it does, `ground` applies the same four checks to it, and if its quotation is not verbatim (models paraphrase), the reading is refused as `quote not found in note`. That refusal is not a bug in the harness; it is the contract doing its job, and the fix belongs in the prompt or in a normalising step that is itself recorded. Print the endpoint and date with anything you quote from this cell.

## 11. Done

Thirteen fixtures pass, refused requests never reach a model, and the reader is a component the contract does not trust. Chapter 3 builds the loop that runs these contracts step by step, with explicit `blocked`, `failed`, and `cancelled` states.

In [ ]:
for fx in FIXTURES:
    calls = []
    def counted(context, m=fx.get("model", fake_reader)):
        calls.append(context); return m(context)
    got = prepare(fx["request"], model=counted)
    seen = {k: v[0] for k, v in got.get("criteria", {}).items()}
    assert all(got.get(k) == v for k, v in fx["expect"].items()), fx["id"]
    assert seen == fx.get("criteria", {}), fx["id"]
print(f"{len(FIXTURES)} fixtures pass.")

## References

- [Yao et al., Tau-bench](https://arxiv.org/abs/2406.12045): domain policies and repeat reliability as evaluation method.
- [Anthropic, Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents): tasks, graders, and outcome checks.

The manifest, notes, fixtures, code, and exercises are original companion material.